In [ ]:
import random
import time

import cv2
import mediapipe as mp
import numpy as np
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
from ugot import ugot

got = ugot.UGOT()
got.initialize("192.168.1.28")
got.show_light_rgb([1, 2, 3], 255, 255, 255)

got.load_models(['gesture'])

## UGOT Gesture Recognition
The UGOT model is able to recognize these gestures:
- Stone
- Scissor
- Cloth
- OK
- Likes

In [ ]:
results = ['Stone', 'Scissor', 'Cloth', 'OK', 'Likes']
print(got.get_gesture_result())

## Rock, Paper, Scissors Game using Gesture Recognition
Create a program to play a rock paper scissor game with the UGOT robot

Some things to take note:
- Gesture names must match with what `got.get_gesture_result()` returns
- The program must randomly select a gesture for the robot (maybe there is a python library that might be able to do this?)
- You must account for all different winning or losing combinations

In [ ]:
# TODO: Write your answer here

## UGOT Movement Control using Gesture Recognition

In [ ]:
# ==========================================
# GESTURE AND DIRECTION MAPPING
# ==========================================
directions = {
    "OK": "forward",
    "Stone": "backward",
    "Scissor": "left",
    "Cloth": "right",
}

got.open_camera()

try:
    while True:
        # Grab the latest camera frame as raw bytes.
        frame = got.read_camera_data()

        if frame:
            # Decode the bytes into an OpenCV image array.
            nparr = np.frombuffer(frame, np.uint8)
            img = cv2.imdecode(nparr, cv2.IMREAD_COLOR)

            if img is not None:
                cv2.imshow("Robot Camera", img)

        # Process window events and check for Q before moving.
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break

        # ==========================================
        # GESTURE RECOGNITION AND UGOT MOVEMENT LOGIC
        # ==========================================

except KeyboardInterrupt:
    print("Remote control ended.")

finally:
    got.mecanum_stop()
    cv2.destroyAllWindows()

## Gesture Recognition with Mediapipe

In [ ]:
directions = {
    "Stone": "forward",
    "Cloth": "backward",
    "Point Left": "left",
    "Point Right": "right",
}

# ==========================================
# CLASSIFY GESTURE BASED ON FINGER POSITION
# ==========================================


# ==========================================
# CALCULATES FINGER ANGLES FROM HAND LANDMARKS
# ==========================================


hand_chains = (
    (0, 1, 2, 3, 4), (0, 5, 6, 7, 8), (5, 9, 10, 11, 12),
    (9, 13, 14, 15, 16), (13, 17, 18, 19, 20), (0, 17),
)

options = vision.HandLandmarkerOptions(
    base_options=python.BaseOptions(
        model_asset_path="hand_landmarker.task"
    ),
    running_mode=vision.RunningMode.VIDEO,
    num_hands=2,
    min_hand_detection_confidence=0.7,
    min_hand_presence_confidence=0.7,
    min_tracking_confidence=0.7,
)

last_timestamp = -1
cap = cv2.VideoCapture(0)

try:
    if not cap.isOpened():
        raise RuntimeError("Could not open webcam")
    
    with vision.HandLandmarker.create_from_options(options) as landmarker:
        while True:
            ret, frame = cap.read()
            
            if not ret:
                print("Camera frame unavailable. Stopping program")
                break
            
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            mp_image = mp.Image(
                image_format=mp.ImageFormat.SRGB,
                data=rgb,
            )

            timestamp = max(
                last_timestamp + 1,
                time.monotonic_ns() // 1_000_000,
            )
            last_timestamp = timestamp

            result = landmarker.detect_for_video(mp_image, timestamp)

            gesture_result = None
            height, width = frame.shape[:2]
            for hand_index, landmarks in enumerate(result.hand_landmarks):
                image_points = [
                    (int(point.x * width), int(point.y * height))
                    for point in landmarks
                ]
                for chain in hand_chains:
                    for start, end in zip(chain, chain[1:]):
                        cv2.line(frame, image_points[start], image_points[end],
                                 (0, 255, 0), 2)
                for point in image_points:
                    cv2.circle(frame, point, 4, (0, 0, 255), -1)

                if len(result.hand_world_landmarks) == 1:
                    gesture_result = recognize_gesture(
                        result.hand_world_landmarks[hand_index], image_points
                    )

            direction = directions.get(gesture_result, "stop")
            label = f"{gesture_result or 'Unknown'}: {direction}"

            cv2.putText(
                frame, label, (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2,
            )
            cv2.imshow("Camera Feed", frame)

            if cv2.waitKey(10) & 0xFF == ord("q"):
                break

            if direction == "forward":
                got.mecanum_move_speed(0, 20)
            elif direction == "backward":
                got.mecanum_move_speed(1, 20)
            elif direction == "left":
                got.mecanum_turn_speed(2, 20)
            elif direction == "right":
                got.mecanum_turn_speed(3, 20)
            else:
                got.mecanum_stop()

except KeyboardInterrupt:
    print("Remote control ended.")

finally:
    try:
        got.mecanum_stop()
    finally:
        cap.release()
        cv2.destroyAllWindows()